# Energy balance of the unit

Heat demand, heat supplied by burning the tar, the char and the gas in that order, and net electricity from the remaining gas (Section 5.1, Eqs. 63 to 65). Worked example S4 at the central values of the parameters, followed by the Monte Carlo over the ranges of Table 1.

In [ ]:
import os
import shutil
import sys
import tempfile
from pathlib import Path
ARTICLE = Path.cwd().parent / "results"
OUTPUT = Path.cwd() / "output" / "07_energy"
OUTPUT.mkdir(parents=True, exist_ok=True)
if (ARTICLE / "best_vectors.npz").exists():
    handle, temporary = tempfile.mkstemp(dir=OUTPUT, suffix=".npz")
    os.close(handle)
    shutil.copyfile(ARTICLE / "best_vectors.npz", temporary)
    os.replace(temporary, OUTPUT / "best_vectors.npz")
os.environ.setdefault("CCPYRO_RESULTS", str(OUTPUT))
os.environ.setdefault("CCPYRO_FIGDATA", str(OUTPUT / "figdata"))
try:
    import ccpyro  # installed with `pip install -e .`
except ImportError:  # running from a clone without installation
    sys.path.insert(0, str(Path.cwd().parent / "src"))
    import ccpyro
import numpy as np
import pandas as pd
np.set_printoptions(precision=6, suppress=True)
print("ccpyro", ccpyro.__version__)

In [ ]:
from ccpyro import energy as en
yields = pd.read_csv(ARTICLE / "product_yields.csv")
feeds = pd.read_csv(ARTICLE / "feedstock_characterisation.csv").set_index("treatment")
central = {k: 0.5 * (a + b) for k, (a, b) in en.RANGES.items()}
row = yields[(yields.treatment == "rice husk (pelletized)") & (yields["T"] == 500.0)].iloc[0]
pd.Series(en.unit_balance(row, feeds.loc["rice husk (pelletized)"], central)).round(4)

In [ ]:
rng = np.random.default_rng(20261014)
rows = []
for treatment in ("rice husk (pelletized)", "corncob (pelletized)"):
    for T in (400.0, 500.0, 600.0, 700.0, 800.0):
        e = en.electricity_draws(yields, feeds, treatment, T, 1000, rng)
        rows.append({"treatment": treatment, "T": T, "kWh/t median": np.median(e),
                     "p05": np.quantile(e, 0.05), "p95": np.quantile(e, 0.95)})
pd.DataFrame(rows).round(0)